# Первая часть лабораторной работы №1

### Установка Spark

Установка зависимостей:
```bash
sudo apt update
sudo apt install -y wget curl tar
```

Скачивание Spark:
```bash
cd ~
wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
```

Распаковка:
```bash
tar -xvzf spark-3.5.1-bin-hadoop3.tgz
mv spark-3.5.1-bin-hadoop3 spark
```

Задание переменных среды:
```bash
echo 'export SPARK_HOME=~/spark' >> ~/.bashrc
echo 'export PATH=$SPARK_HOME/bin:$PATH' >> ~/.bashrc
source ~/.bashrc
```

Проверка:
```bash
spark-shell
```
или
```bash
pyspark
```

In [106]:
from pyspark import SparkContext
sc = SparkContext("local[*]", "lab")

### Создание Resilient Distributed Dataset

In [107]:
warandpeace = sc.textFile("/mnt/c/Users/artem/Documents/BigData/Lab1/data/warandsociety.txt")

Выведите количество строк файла.

In [108]:
warandpeace.count()

12851

Попробуйте считать несуществующий файл, например nil, а затем вывести количество его строк на экран

In [109]:
nilFile = sc.textFile("nil")
nilFile.count()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/mnt/c/Users/artem/Documents/BigData/Lab1/L1 - Introduction to Apache Spark/nil
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:210)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:290)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.s

Считайте первые 10 строк файла warandsociety.txt.

In [110]:
warandpeace.take(10)

['Лев Николаевич Толстой',
 'Война и мир. Книга 1',
 '',
 'Война и мир – 1',
 '',
 ' ',
 ' http://www.lib.ru',
 '',
 'Аннотация ',
 '']

Узнайте на сколько частей разделились данные в кластере.

In [111]:
warandpeace.getNumPartitions()

2

Создайте распределённую коллекцию из нескольких элементов и для каждого элемента верните ip адрес, на котором он расположен:

In [112]:
sc.parallelize([1,2,3]).map(lambda x: __import__("socket").gethostname()).collect()

['artmkchmv', 'artmkchmv', 'artmkchmv']

Найдите строки, в которых содержится слово "война". Запросите первую строку. Строкой в данном файле является целый абзац, так как только по завершению абзаца содержится символ переноса строки.

In [113]:
linesWithWar = warandpeace.filter(lambda x: "война" in x)
linesWithWar.first()

"– Еh bien, mon prince. Genes et Lucques ne sont plus que des apanages, des поместья, de la famille Buonaparte. Non, je vous previens, que si vous ne me dites pas, que nous avons la guerre, si vous vous permettez encore de pallier toutes les infamies, toutes les atrocites de cet Antichrist (ma parole, j'y crois) – je ne vous connais plus, vous n'etes plus mon ami, vous n'etes plus мой верный раб, comme vous dites. [Ну, что, князь, Генуа и Лукка стали не больше, как поместьями фамилии Бонапарте. Нет, я вас предупреждаю, если вы мне не скажете, что у нас война, если вы еще позволите себе защищать все гадости, все ужасы этого Антихриста (право, я верю, что он Антихрист) – я вас больше не знаю, вы уж не друг мой, вы уж не мой верный раб, как вы говорите.] Ну, здравствуйте, здравствуйте. Je vois que je vous fais peur, [Я вижу, что я вас пугаю,] садитесь и рассказывайте."

Воспользуйтесь следующим блоком кода для замера времени выполнения команды.

In [114]:
import time

def timer(f):
    t0 = time.time()
    r = f()
    print("Elapsed:", time.time() - t0)
    return r

In [115]:
linesWithWar.cache()

timer(lambda: linesWithWar.count())
timer(lambda: linesWithWar.count())

Elapsed: 0.21638154983520508
Elapsed: 0.10768842697143555


54

Найдите гистограмму слов.

In [116]:
wordCounts = (
    linesWithWar
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

In [117]:
wordCounts.take(10)

[('Еh', 1),
 ('prince.', 1),
 ('et', 4),
 ('sont', 1),
 ('plus', 3),
 ('des', 2),
 ('la', 6),
 ('Buonaparte.', 1),
 ('Non,', 2),
 ('vous', 10)]

Упражнение. Улучшите процедуру, убирая из слов лишние символы и трансформируя все слова в нижний регистр. Используйте регулярные выражения. Например, по регулярному выражению "\w*".r следующий код

In [118]:
import re

def clean(word):
    return re.sub(r"[^\wа-яА-Яa-zA-Z]", "", word.lower())

In [119]:
wordCountsClean = (
    warandpeace
    .flatMap(lambda line: line.split(" "))
    .map(clean)
    .filter(lambda x: x != "")
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
)

Сохраните результаты в файл, а затем, найдите данные в HDFS и выведите данные в linux консоли с помощью команды 
```bash
hadoop fs -cat warandpeace_histogram.txt/* 
```
(здесь используется относительный путь).

In [120]:
wordCountsClean.saveAsTextFile("warandpeace_histogram.txt")

### Операции с множествами

Инициализируйте два множества.

In [121]:
a = sc.parallelize([1,2,3,4])
b = sc.parallelize([3,4,6,7])

Найдите объединение a и b и соберите данные на главный узел с помощью функции collect.

In [122]:
a.union(b).collect()

[1, 2, 3, 4, 3, 4, 6, 7]

Уберите дубликаты с помощью distinct.

In [123]:
a.union(b).distinct().collect()

[1, 2, 3, 4, 6, 7]

Найдите пересечение множеств.

In [124]:
a.intersection(b).collect()

[3, 4]

Найдите разность множеств.

In [125]:
a.subtract(b).collect()

[1, 2]

### Общие переменные

#### Широковещательные переменные

Cоздайте широковещательную переменную. Для получения значения обратитесь к полю value.

In [126]:
broadcastVar = sc.broadcast([1,2,3])
broadcastVar.value

[1, 2, 3]

#### Аккумулирующие переменные

Потренируйтесь в создании аккумулирующих переменных.

In [127]:
accum = sc.accumulator(0)

Следующим шагом запустите параллельную обработку массива и в каждом параллельном задании добавьте к аккумулирующей переменной значение элемента массива. Для получения текущего значения вызовите команду.

In [128]:
sc.parallelize([1,2,3,4]).foreach(lambda x: accum.add(x))
accum.value

10

### Пары ключ-значение

Создайте пару ключ-значение из двух букв. Для доступа к первому значению обратитесь к полю _1, для доступа к второму значению к полю _2.

In [129]:
pair = ('a', 'b')
pair[0], pair[1]

('a', 'b')

### Топ-10 популярных номеров такси

Создайте RDD на основе загруженных данных nyctaxi.csv.

In [130]:
taxi = sc.textFile("/mnt/c/Users/artem/Documents/BigData/Lab1/data/nyctaxi.csv")

Выведите первые 5 строк из данной таблицы.

In [131]:
taxi.take(5)

['"_id","_rev","dropoff_datetime","dropoff_latitude","dropoff_longitude","hack_license","medallion","passenger_count","pickup_datetime","pickup_latitude","pickup_longitude","rate_code","store_and_fwd_flag","trip_distance","trip_time_in_secs","vendor_id"',
 '"29b3f4a30dea6688d4c289c9672cb996","1-ddfdec8050c7ef4dc694eeeda6c4625e","2013-01-11 22:03:00",+4.07033460000000E+001,-7.40144200000000E+001,"A93D1F7F8998FFB75EEF477EB6077516","68BC16A99E915E44ADA7E639B4DD5F59",2,"2013-01-11 21:48:00",+4.06760670000000E+001,-7.39810790000000E+001,1,,+4.08000000000000E+000,900,"VTS"',
 '"2a80cfaa425dcec0861e02ae44354500","1-b72234b58a7b0018a1ec5d2ea0797e32","2013-01-11 04:28:00",+4.08190960000000E+001,-7.39467470000000E+001,"64CE1B03FDE343BB8DFB512123A525A4","60150AA39B2F654ED6F0C3AF8174A48A",1,"2013-01-11 04:07:00",+4.07280540000000E+001,-7.40020370000000E+001,1,,+8.53000000000000E+000,1260,"VTS"',
 '"29b3f4a30dea6688d4c289c96758d87e","1-387ec30eac5abda89d2abefdf947b2c1","2013-01-11 22:02:00",+4.0727

Обратите внимание, что первая строка является заголовком. Её как правило нужно будет отфильтровать.

In [132]:
header = taxi.first()
data = taxi.filter(lambda x: x != header)

Для разбора значений потребуется создать RDD, где каждая строка разбита на массив подстрок. Используйте запятую в качестве разделителя.

In [133]:
taxiParse = data.map(lambda x: x.split(","))

Теперь преобразуем массив строк в массив пар ключ-значение, где ключом будет служить номер такси (6 колонка), а значением единица.

In [134]:
taxiMedKey = taxiParse.map(lambda r: (r[6], 1))

Следом мы можем найти количество поездок каждого номера такси.

In [135]:
taxiCounts = taxiMedKey.reduceByKey(lambda a,b: a+b)

Выведем полученные результаты в отсортированном виде.

In [136]:
top10 = taxiCounts.map(lambda x: (x[1], x[0])).top(10)
[(v, k) for k, v in top10]

[('"AB44AD9A03B7CFAF3925103BDCC0AF23"', 44),
 ('"71CACFBADF9568AAE88A843DB511D172"', 41),
 ('"6483B9BFCB216EC88986EA3AB13064E7"', 41),
 ('"4C73459B430339981D78795300433438"', 41),
 ('"67E71D24AF704D814A0A825005ADA72E"', 40),
 ('"02E5A4136FD0A775A023A005A4EABC62"', 40),
 ('"9DFBCD218E7116F34C044F0680A0FB8A"', 39),
 ('"8DEB70907D00AA1D7FF5E2683240549B"', 39),
 ('"7989C2AB3F345F4AB54D3CF1E0480D67"', 39),
 ('"6C9F67DF658DC5636F9E7752F203F70A"', 39)]

Попробуйте найти общее количество номеров такси несколько раз, предварительно объявив RDD taxiCounts как сохраняемую в кэше.

In [137]:
taxiCounts.cache()
timer(lambda: taxiCounts.count())
timer(lambda: taxiCounts.count())

Elapsed: 0.20736479759216309
Elapsed: 0.10592961311340332


13370

In [138]:
import pyspark
taxi.persist(storageLevel=pyspark.StorageLevel.MEMORY_ONLY)

/mnt/c/Users/artem/Documents/BigData/Lab1/data/nyctaxi.csv MapPartitionsRDD[53] at textFile at NativeMethodAccessorImpl.java:0

In [139]:
sc.stop()

# Вторая часть лабораторной работы №1

## 1. Проведите анализ данных велопарковок на языке Python в интерактивном режиме из Jupyter книг:
  - `L1_interactive_bike_analysis_python_with_rdd.ipynb`

### Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark

#### Описание данных

https://www.kaggle.com/benhamner/sf-bay-area-bike-share

#### stations.csv схема:

```
id: station ID number
name: name of station
lat: latitude
long: longitude
dock_count: number of total docks at station
city: city (San Francisco, Redwood City, Palo Alto, Mountain View, San Jose)
installation_date: original date that station was installed. If station was moved, it is noted below.
```

#### trips.csv схема:

```
id: numeric ID of bike trip
duration: time of trip in seconds
start_date: start date of trip with date and time, in PST
start_station_name: station name of start station
start_station_id: numeric reference for start station
end_date: end date of trip with date and time, in PST
end_station_name: station name for end station
end_station_id: numeric reference for end station
bike_id: ID of bike used
subscription_type: Subscriber = annual or 30-day member; Customer = 24-hour or 3-day member
zip_code: Home zip code of subscriber (customers can choose to manually enter zip at kiosk however data is unreliable)
```

In [140]:
from pathlib import Path
import os
from pyspark import SparkContext, SparkConf

def lab_data_dir():
    for d in (Path("data"), Path("../data")):
        if (d / "trips.csv").exists():
            return d.resolve()
    raise FileNotFoundError("Нет trips.csv в ./data или ../data (запускайте из корня Lab1 или из папки L1)")

DATA_DIR = lab_data_dir()
TRIPS_CSV = str(DATA_DIR / "trips.csv")
STATIONS_CSV = str(DATA_DIR / "stations.csv")

In [141]:
_master = os.environ.get("SPARK_MASTER", "local[*]")
conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster(_master)

In [142]:
sc = SparkContext(conf=conf)

In [143]:
tripData = sc.textFile(TRIPS_CSV)
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile(STATIONS_CSV)
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))

In [144]:
list(enumerate(tripsHeader.split(",")))

[(0, 'id'),
 (1, 'duration'),
 (2, 'start_date'),
 (3, 'start_station_name'),
 (4, 'start_station_id'),
 (5, 'end_date'),
 (6, 'end_station_name'),
 (7, 'end_station_id'),
 (8, 'bike_id'),
 (9, 'subscription_type'),
 (10, 'zip_code')]

In [145]:
list(enumerate(stationsHeader.split(",")))

[(0, 'id'),
 (1, 'name'),
 (2, 'lat'),
 (3, 'long'),
 (4, 'dock_count'),
 (5, 'city'),
 (6, 'installation_date')]

In [146]:
trips.take(2)

[['4576',
  '63',
  '',
  'South Van Ness at Market',
  '66',
  '8/29/2013 14:14',
  'South Van Ness at Market',
  '66',
  '520',
  'Subscriber',
  '94127'],
 ['4607',
  '',
  '8/29/2013 14:42',
  'San Jose City Hall',
  '10',
  '8/29/2013 14:43',
  'San Jose City Hall',
  '10',
  '661',
  'Subscriber',
  '95138']]

In [147]:
stations.take(2)

[['2',
  'San Jose Diridon Caltrain Station',
  '37.329732',
  '-121.90178200000001',
  '27',
  'San Jose',
  '8/6/2013'],
 ['3',
  'San Jose Civic Center',
  '37.330698',
  '-121.888979',
  '15',
  'San Jose',
  '8/5/2013']]

Объявите stationsIndexed так, чтобы результатом был список пар ключ-значение с целочисленным ключом из первой колонки. Таким образом вы создаёте индекс на основе первой колонки - номера велостоянки.

In [148]:
stationsIndexed = stations.keyBy(lambda station: station[0])

In [149]:
stationsIndexed.take(2)

[('2',
  ['2',
   'San Jose Diridon Caltrain Station',
   '37.329732',
   '-121.90178200000001',
   '27',
   'San Jose',
   '8/6/2013']),
 ('3',
  ['3',
   'San Jose Civic Center',
   '37.330698',
   '-121.888979',
   '15',
   'San Jose',
   '8/5/2013'])]

Аналогичное действие проделайте для индексирования коллекции trips по колонкам start_station_id и end_station_id и сохраните результат в переменные, например tripsByStartTerminals и tripsByEndTerminals.

In [150]:
tripsByStartTerminals = trips.keyBy(lambda trip: trip[4])
tripsByEndTerminals = trips.keyBy(lambda trip: trip[7])

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

In [151]:
startTrips = stationsIndexed.join(tripsByStartTerminals)
endTrips = stationsIndexed.join(tripsByEndTerminals)

Объявление последовательности трансформаций приводит к созданию ацикличного ориентированного графа. Вывести полученный граф можно для любого RDD.

In [152]:
print(startTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[23] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[15] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[14] at partitionBy at <unknown>:0 []
 +-(5) PairwiseRDD[13] at join at /tmp/ipykernel_3808/210173577.py:1 []
    |  PythonRDD[12] at join at /tmp/ipykernel_3808/210173577.py:1 []
    |  UnionRDD[11] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[9] at RDD at PythonRDD.scala:53 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/stations.csv MapPartitionsRDD[4] at textFile at <unknown>:0 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/stations.csv HadoopRDD[3] at textFile at <unknown>:0 []
    |  PythonRDD[10] at RDD at PythonRDD.scala:53 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/trips.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/trips.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


In [153]:
print(endTrips.toDebugString().decode("utf-8"))

(5) PythonRDD[24] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[22] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[21] at partitionBy at <unknown>:0 []
 +-(5) PairwiseRDD[20] at join at /tmp/ipykernel_3808/210173577.py:2 []
    |  PythonRDD[19] at join at /tmp/ipykernel_3808/210173577.py:2 []
    |  UnionRDD[18] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[16] at RDD at PythonRDD.scala:53 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/stations.csv MapPartitionsRDD[4] at textFile at <unknown>:0 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/stations.csv HadoopRDD[3] at textFile at <unknown>:0 []
    |  PythonRDD[17] at RDD at PythonRDD.scala:53 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/trips.csv MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /mnt/c/Users/artem/Documents/BigData/Lab1/data/trips.csv HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


Выполните объявленные графы трансформаций вызовом команды count.

In [154]:
startTrips.count()

669959

In [155]:
endTrips.count()

669959

Если вы знаете распределение ключей заранее, вы можете выбрать оптимальный способ хеширования ключей по разделам Partition. Например, если один ключ встречается на порядки чаще, чем другие ключи, то использование HashPartitioner будет не лучшим выбором, так как данные связанные с этим ключом будут собираться в одном разделе. Это приведёт к неравномерной нагрузке на вычислительные ресурсы.

Выбрать определённую реализацию класса распределения по разделам можно с помощью функции RDD partitionBy. Например, для RDD stationsIndexed выбирается portable_hash(idx) с количеством разделов равным количеству разделов trips RDD.

In [156]:
from pyspark.rdd import portable_hash

stationsIndexed.partitionBy(numPartitions=trips.getNumPartitions(), partitionFunc=lambda x: portable_hash(x[0]))

MapPartitionsRDD[30] at mapPartitions at PythonRDD.scala:160

Узнать какой класс назначен для текущего RDD можно обращением к полю partitioner.

In [157]:
stationsIndexed.partitioner

#### Создание модели данных

Для более эффективной  обработки и получения дополнительных возможностей мы можем объявить классы сущностей предметной области и преобразовать исходные строковые данные в объявленное представление.

In [158]:
from typing import NamedTuple
from datetime import datetime

In [159]:
def initStation(stations):
    class Station(NamedTuple):
        station_id: int
        name: str
        lat: float
        long: float
        dockcount: int
        landmark: str
        installation: str
    
    for station in stations:
        yield Station(
            station_id = int(station[0]),
            name = station[1],
            lat = float(station[2]),
            long = float(station[3]),
            dockcount = int(station[4]),
            landmark = station[5],
            installation = datetime.strptime(station[6], '%m/%d/%Y')
        )

In [160]:
stationsInternal = stations.mapPartitions(initStation)
stationsInternal.first()

Station(station_id=2, name='San Jose Diridon Caltrain Station', lat=37.329732, long=-121.90178200000001, dockcount=27, landmark='San Jose', installation=datetime.datetime(2013, 8, 6, 0, 0))

In [161]:
def initTrip(trips):
    class Trip(NamedTuple):
        trip_id: int
        duration: int
        start_date: datetime
        start_station_name: str
        start_station_id: int
        end_date: datetime
        end_station_name: str
        end_station_id: int
        bike_id: int
        subscription_type: str
        zip_code: str
        
    for trip in trips:
        try:
            yield Trip(                             
             trip_id = int(trip[0]),
             duration = int(trip[1]),
             start_date = datetime.strptime(trip[2], '%m/%d/%Y %H:%M'),
             start_station_name = trip[3],
             start_station_id = int(trip[4]),
             end_date = datetime.strptime(trip[5], '%m/%d/%Y %H:%M'),
             end_station_name = trip[6],
             end_station_id = int(trip[7]),
             bike_id = int(trip[8]),
             subscription_type = trip[9],
             zip_code = trip[10]
            ) 
        except:
            pass

In [162]:
tripsInternal = trips.mapPartitions(initTrip)
tripsInternal.take(10)

Exception ignored in: <generator object initTrip at 0x713e74415b90>
Traceback (most recent call last):
  File "/mnt/c/Users/artem/Documents/BigData/Lab1/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
RuntimeError: generator ignored GeneratorExit


[Trip(trip_id=4130, duration=71, start_date=datetime.datetime(2013, 8, 29, 10, 16), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 10, 17), end_station_name='Mountain View City Hall', end_station_id=27, bike_id=48, subscription_type='Subscriber', zip_code='97214'),
 Trip(trip_id=4251, duration=77, start_date=datetime.datetime(2013, 8, 29, 11, 29), start_station_name='San Jose City Hall', start_station_id=10, end_date=datetime.datetime(2013, 8, 29, 11, 30), end_station_name='San Jose City Hall', end_station_id=10, bike_id=26, subscription_type='Subscriber', zip_code='95060'),
 Trip(trip_id=4299, duration=83, start_date=datetime.datetime(2013, 8, 29, 12, 2), start_station_name='South Van Ness at Market', start_station_id=66, end_date=datetime.datetime(2013, 8, 29, 12, 4), end_station_name='Market at 10th', end_station_id=67, bike_id=319, subscription_type='Subscriber', zip_code='94103'),
 Trip(trip_id=4927, duration=103, start_d

Для каждой стартовой станции найдем среднее время поездки. Будем использовать метод groupByKey.

Для этого потребуется преобразовать trips RDD в RDD коллекцию пар ключ-значение аналогично тому, как мы совершали это ранее методом keyBy.

In [163]:
tripsByStartStation = tripsInternal.keyBy(lambda trip: trip.start_station_name)

Рассчитаем среднее время поездки для каждого стартового парковочного места.

In [164]:
import numpy as np

avgDurationByStartStation = tripsByStartStation\
 .mapValues(lambda trip: trip.duration)\
 .groupByKey()\
 .mapValues(lambda trip_durations: np.mean(list(trip_durations)))

Выведем первые 10 результатов.


In [165]:
%%time

avgDurationByStartStation.top(10, key=lambda x: x[1])

[Stage 11:===================>                                      (1 + 2) / 3]

CPU times: user 2.46 ms, sys: 8.38 ms, total: 10.8 ms
Wall time: 3.23 s


[('University and Emerson', np.float64(7090.239417989418)),
 ('California Ave Caltrain Station', np.float64(4628.005847953216)),
 ('Redwood City Public Library', np.float64(4579.234741784037)),
 ('Park at Olive', np.float64(4438.1613333333335)),
 ('San Jose Civic Center', np.float64(4208.016938519448)),
 ('Rengstorff Avenue / California Street', np.float64(4174.082373782108)),
 ('Redwood City Medical Center', np.float64(3959.491961414791)),
 ('Palo Alto Caltrain Station', np.float64(3210.6489815253435)),
 ('San Mateo County Center', np.float64(2716.7700348432054)),
 ('Broadway at Main', np.float64(2481.2537313432836))]

Выполнение операции groupByKey приводит к интенсивным передачам данных. Если группировка делается для последующей редукции элементов лучше использовать трансформацию reduceByKey или aggregateByKey. Их выполнение приведёт сначала к локальной редукции над разделом Partition, а затем будет произведено окончательное суммирование над полученными частичными суммами.

Примечание. Выполнение reduceByKey логически сходно с выполнением Combine и Reduce фазы MapReduce работы.

Функция aggregateByKey является аналогом reduceByKey с возможностью указывать начальный элемент.

Рассчитаем среднее значение с помощью aggregateByKey. Одновременно будут вычисляться два значения для каждого стартового терминала: сумма времён и количество поездок.

In [166]:
? tripsByStartStation.aggregateByKey

Signature:
 tripsByStartStation.aggregateByKey(
    zeroValue: ~U,
    seqFunc: Callable[[~U, ~V], ~U],
    combFunc: Callable[[~U, ~U], ~U],
    numPartitions: Optional[int] = None,
    partitionFunc: Callable[[~K], int] = <function portable_hash at 0x7b5db2fe42c0>,
) -> 'RDD[Tuple[K, U]]'
Docstring:
Aggregate the values of each key, using given combine functions and a neutral
"zero value". This function can return a different result type, U, than the type
of the values in this RDD, V. Thus, we need one operation for merging a V into
a U and one operation for merging two U's, The former operation is used for merging
values within a partition, and the latter is used for merging values between
partitions. To avoid memory allocation, both of these functions are
allowed to modify and return their first argument instead of creating a new U.

.. versionadded:: 1.1.0

Parameters
----------
zeroValue : U
    the initial value for the accumulated result of each partition
seqFunc : function
   

In [167]:
def seqFunc(acc, duration):
    duration_sum, count = acc
    return (duration_sum + duration, count + 1)

def combFunc(acc1, acc2):
    duration_sum1, count1 = acc1
    duration_sum2, count2 = acc2
    return (duration_sum1+duration_sum2, count1+count2)

def meanFunc(acc):
    duration_sum, count = acc
    return duration_sum/count

avgDurationByStartStation2 = tripsByStartStation\
  .mapValues(lambda trip: trip.duration)\
  .aggregateByKey(
    zeroValue=(0,0),
    seqFunc=seqFunc,
    combFunc=combFunc)\
  .mapValues(meanFunc)

В zeroValue передаётся начальное значение. В нашем случае это пара нулей. Первая функция seqFunc предназначена для прохода по коллекции партиции. На этом проходе значение элементов помещаются средой в переменную duration, а переменная «аккумулятора» acc накапливает значения. Вторая функция combFunc предназначена для этапа редукции частично посчитанных локальных результатов.

Сравните результаты avgDurationByStartStation и avgDurationByStartStation2 и их время выполнения.

In [168]:
%%time

avgDurationByStartStation2.top(10, key=lambda x: x[1])

[Stage 13:===================>                                      (1 + 2) / 3]

CPU times: user 1.46 ms, sys: 7.93 ms, total: 9.39 ms
Wall time: 3.21 s


[('University and Emerson', 7090.239417989418),
 ('California Ave Caltrain Station', 4628.005847953216),
 ('Redwood City Public Library', 4579.234741784037),
 ('Park at Olive', 4438.1613333333335),
 ('San Jose Civic Center', 4208.016938519448),
 ('Rengstorff Avenue / California Street', 4174.082373782108),
 ('Redwood City Medical Center', 3959.491961414791),
 ('Palo Alto Caltrain Station', 3210.6489815253435),
 ('San Mateo County Center', 2716.7700348432054),
 ('Broadway at Main', 2481.2537313432836)]

Теперь найдём первую поездку для каждой велостоянки. Для решения опять потребуется группировка. Ещё одним недостатком groupByKey данных является то, что для группировки данные должны поместиться в оперативной памяти. Это может привести к ошибке OutOfMemoryException для больших объёмов данных.

Найдем самую раннюю поездку для каждой станции. Сгруппируем поездки по станциям, возьмём первую поездку из отсортированного списка:

In [169]:
def earliestTrip(trips):
    if trips is None:
        return None
    if len(trips)==0:
        return trips
    trips = list(trips)
    min_date = trips[0].start_date
    min_trip = trips[0]
    for trip in trips[1:]:
        if min_date > trip.start_date:
            min_date = trip.start_date
            min_trip = trip
    return min_trip

firstGrouped = tripsByStartStation\
  .groupByKey()\
  .mapValues(lambda trips: earliestTrip(trips))

In [170]:
%%time

firstGrouped.take(5)

[Stage 15:===================>                                      (1 + 2) / 3]

CPU times: user 8.96 ms, sys: 0 ns, total: 8.96 ms
Wall time: 5.87 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id=27, bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id=5, bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id=10, bike_id=661, su

Лучшим вариантом с точки зрения эффективности будет использование трансформации reduceByKey.

In [171]:
firstGrouped = tripsByStartStation\
  .reduceByKey(lambda tripA, tripB: tripA if tripA.start_date < tripB.start_date else tripB)

In [172]:
%%time

firstGrouped.take(5)

[Stage 17:===================>                                      (1 + 2) / 3]

CPU times: user 10.6 ms, sys: 572 μs, total: 11.2 ms
Wall time: 3.27 s


[('Mountain View City Hall',
  Trip(trip_id=4081, duration=218, start_date=datetime.datetime(2013, 8, 29, 9, 38), start_station_name='Mountain View City Hall', start_station_id=27, end_date=datetime.datetime(2013, 8, 29, 9, 41), end_station_name='Mountain View City Hall', end_station_id=27, bike_id=150, subscription_type='Subscriber', zip_code='97214')),
 ('Santa Clara at Almaden',
  Trip(trip_id=4500, duration=109, start_date=datetime.datetime(2013, 8, 29, 13, 25), start_station_name='Santa Clara at Almaden', start_station_id=4, end_date=datetime.datetime(2013, 8, 29, 13, 27), end_station_name='Adobe on Almaden', end_station_id=5, bike_id=679, subscription_type='Subscriber', zip_code='95112')),
 ('San Salvador at 1st',
  Trip(trip_id=4517, duration=379, start_date=datetime.datetime(2013, 8, 29, 13, 34), start_station_name='San Salvador at 1st', start_station_id=8, end_date=datetime.datetime(2013, 8, 29, 13, 41), end_station_name='San Jose City Hall', end_station_id=10, bike_id=661, su

In [173]:
sc.stop()

  - `L1_interactive_bike_analysis_python_with_dataframes.ipynb`.

In [174]:
from pathlib import Path
import os
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql as sql

def lab_data_dir():
    for d in (Path("data"), Path("../data")):
        if (d / "trips.csv").exists():
            return d.resolve()
    raise FileNotFoundError("Нет trips.csv в ./data или ../data")

DATA_DIR = lab_data_dir()
TRIPS_CSV = str(DATA_DIR / "trips.csv")
STATIONS_CSV = str(DATA_DIR / "stations.csv")

In [175]:
_master = os.environ.get("SPARK_MASTER", "local[*]")
conf = SparkConf().setAppName("L1_interactive_bike_analysis").setMaster(_master)

In [176]:
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

#### Пример чтения csv файлов и работы с дефектными данными

Список опций чтения и записи для CSV файлов https://spark.apache.org/docs/latest/sql-data-sources-csv.html#data-source-option

Формат паттерна временной метки Spark SQL отличается от python библиотеки datetime.
https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

In [177]:
tripData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv(TRIPS_CSV)

tripData

DataFrame[id: int, duration: int, start_date: timestamp, start_station_name: string, start_station_id: int, end_date: timestamp, end_station_name: string, end_station_id: int, bike_id: int, subscription_type: string, zip_code: string]

In [178]:
tripData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [179]:
tripData.show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|               NULL|South Van Ness at...|              66|2013-08-29 14:14:00|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|    NULL|2013-08-29 14:42:00|  San Jose City Hall|              10|2013-08-29 14:43:00|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|    

In [180]:
? tripData.dropna

Signature:
 tripData.dropna(
    how: str = 'any',
    thresh: Optional[int] = None,
    subset: Union[str, Tuple[str, ...], List[str], NoneType] = None,
) -> 'DataFrame'
Docstring:
Returns a new :class:`DataFrame` omitting rows with null values.
:func:`DataFrame.dropna` and :func:`DataFrameNaFunctions.drop` are aliases of each other.

.. versionadded:: 1.3.1

.. versionchanged:: 3.4.0
    Supports Spark Connect.

Parameters
----------
how : str, optional
    'any' or 'all'.
    If 'any', drop a row if it contains any nulls.
    If 'all', drop a row only if all its values are null.
thresh: int, optional
    default None
    If specified, drop rows that have less than `thresh` non-null values.
    This overwrites the `how` parameter.
subset : str, tuple or list, optional
    optional list of column names to consider.

Returns
-------
:class:`DataFrame`
    DataFrame with null only rows excluded.

Examples
--------
>>> from pyspark.sql import Row
>>> df = spark.createDataFrame([
...     

In [181]:
tripData.dropna().show(n=5)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [182]:
tripData.describe().show()

26/05/01 17:29:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:====>                                                    (1 + 11) / 12]

+-------+-----------------+------------------+--------------------+-----------------+--------------------+------------------+------------------+-----------------+--------------------+
|summary|               id|          duration|  start_station_name| start_station_id|    end_station_name|    end_station_id|           bike_id|subscription_type|            zip_code|
+-------+-----------------+------------------+--------------------+-----------------+--------------------+------------------+------------------+-----------------+--------------------+
|  count|           669959|            669958|              669959|           669959|              669959|            669959|            669959|           669959|              663340|
|   mean|460382.0098991132| 1107.951395460611|                NULL|57.85187601032302|                NULL|57.837437813358726|427.58761954089726|             NULL|  1576245.3147059095|
| stddev|264584.4584872584|22255.453593552553|                NULL|17.1124739683

In [183]:
stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv(STATIONS_CSV)

stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [184]:
stationData.show(n=5)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
only showing top 5 rows



In [185]:
stationData.describe().show()

+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|summary|                id|                name|                lat|               long|       dock_count|         city|
+-------+------------------+--------------------+-------------------+-------------------+-----------------+-------------+
|  count|                70|                  70|                 70|                 70|               70|           70|
|   mean|              43.0|                NULL|  37.59024338428572|-122.21841616428571|17.65714285714286|         NULL|
| stddev|24.166091947189145|                NULL|0.20347253639672502|0.20944604979644524|4.010441857493954|         NULL|
|    min|                 2|       2nd at Folsom|          37.329732|        -122.418954|               11|Mountain View|
|    max|                84|Yerba Buena Cente...|           37.80477|        -121.877349|               27|     San Jose|
+-------+---------------

#### Пример использования DataFrame API

Выполните операцию объединения коллекций по ключу с помощью функции join. Объедините stationsIndexed и tripsByStartTerminals, stationsIndexed и tripsByEndTerminals.

In [186]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



In [187]:
stationsView = stationData.select(stationData['id'], stationData['name'], stationData['lat'], stationData['long'])
stationsView.show()

+---+--------------------+------------------+-------------------+
| id|                name|               lat|               long|
+---+--------------------+------------------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|
|  8| San Salvador at 1st|         37.330165|-121.88583100000001|
|  9|           Japantown|         37.348742|-121.89471499999999|
| 10|  San Jose City Hall|         37.337391|        -121.886995|
| 11|         MLK Library|         37.335885|-121.88566000000002|
| 12|SJSU 4th at San C...|         37.332808|-121.88389099999999|
| 13|       St James Park|         37.339301|-121.88993700000002|
| 14|Arena

In [188]:
startTrips = tripData.select(tripData.id, tripData.duration, tripData.start_station_id).withColumnRenamed('id', 'trip_id').join(stationsView, tripData.start_station_id == stationsView.id)
startTrips = startTrips.drop('id')

In [189]:
startTrips.show()

+-------+--------+----------------+--------------------+------------------+-------------------+
|trip_id|duration|start_station_id|                name|               lat|               long|
+-------+--------+----------------+--------------------+------------------+-------------------+
|   4576|      63|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4607|    NULL|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|      71|              27|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|      77|              10|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|      83|              66|South Van Ness at...|         37.774814|        -122.418954|
|   4927|     103|              59| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|     109|               4|Santa Clara at Al...|         37.333988|        -121.894902|
|   4563|     111|               8| San 

In [190]:
tripData.printSchema()
stationData.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: timestamp (nullable = true)



#### Пример использования Spark SQL API

In [191]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

In [192]:
endTrips = spark.sql("""
SELECT trips.id as trip_id, trips.end_station_id, trips.duration, stations.name as station_name, stations.lat, stations.long 
FROM trips INNER JOIN stations 
    ON trips.end_station_id==stations.id
""")

In [193]:
endTrips.show()

+-------+--------------+--------+--------------------+------------------+-------------------+
|trip_id|end_station_id|duration|        station_name|               lat|               long|
+-------+--------------+--------+--------------------+------------------+-------------------+
|   4576|            66|      63|South Van Ness at...|         37.774814|        -122.418954|
|   4607|            10|    NULL|  San Jose City Hall|         37.337391|        -121.886995|
|   4130|            27|      71|Mountain View Cit...|         37.389218|        -122.081896|
|   4251|            10|      77|  San Jose City Hall|         37.337391|        -121.886995|
|   4299|            67|      83|      Market at 10th|37.776619000000004|-122.41738500000001|
|   4927|            59|     103| Golden Gate at Polk|         37.781332|        -122.418603|
|   4500|             5|     109|    Adobe on Almaden|         37.331415|          -121.8932|
|   4563|             8|     111| San Salvador at 1st|      

Для каждой стартовой станции найдем среднее время поездки. Рассчитаем среднее время поездки для каждого стартового парковочного места.

In [194]:
spark.sql("""
SELECT start_station_name, avg(duration)
FROM trips
GROUP BY trips.start_station_name
ORDER BY avg(duration) DESC
""").show()

+--------------------+------------------+
|  start_station_name|     avg(duration)|
+--------------------+------------------+
|University and Em...| 7090.239417989418|
|California Ave Ca...| 4628.005847953216|
|Redwood City Publ...| 4579.234741784037|
|       Park at Olive|4438.1613333333335|
|San Jose Civic Ce...| 4208.016938519448|
|Rengstorff Avenue...| 4174.082373782108|
|Redwood City Medi...| 3959.491961414791|
|Palo Alto Caltrai...|3210.6489815253435|
|San Mateo County ...|2716.7700348432054|
|    Broadway at Main|2481.2537313432836|
|Cowper at University|2477.2379912663755|
|Redwood City Calt...|2415.7175032175032|
|South Van Ness at...| 2401.603338509317|
|San Antonio Caltr...| 2377.497487437186|
|San Antonio Shopp...| 2285.981298129813|
|           Japantown| 2207.809947643979|
|Washington at Kea...|1979.3077445652175|
|Arena Green / SAP...|1963.9679144385027|
|SJSU 4th at San C...| 1962.280341880342|
|Castro Street and...|1868.3135135135135|
+--------------------+------------

#### Пример подготовки данных c Spark SQL, pandas, h3 для их визуализации на карте folium

In [195]:
# ! pip install h3 h3_pyspark pandas folium

Найдём велосипеды, которые ездили в рождество 2014 года.

In [196]:
# year - the year to represent, from 1 to 9999  
# month - the month-of-year to represent, from 1 (January) to 12 (December)  
# day - the day-of-month to represent, from 1 to 31  
# hour - the hour-of-day to represent, from 0 to 23  
# min - the minute-of-hour to represent, from 0 to 59  
# sec - the second-of-minute and its micro-fraction to represent, from 0 to 60. The value can be either an integer like 13 , or a fraction like 13.123. If the sec argument equals to 60, the seconds field is set to 0 and 1 minute is added to the final timestamp.  
# timezone - the time zone identifier. For example, CET, UTC and etc.

spark.sql("""
SELECT bike_id, start_date, end_date 
FROM trips 
WHERE 
    start_date > make_timestamp(2014, 12, 25, 0, 0, 0) 
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+
|bike_id|         start_date|           end_date|
+-------+-------------------+-------------------+
|    379|2014-12-25 22:10:00|2014-12-25 22:18:00|
|    709|2014-12-25 21:21:00|2014-12-25 21:27:00|
|    376|2014-12-25 20:40:00|2014-12-25 20:46:00|
|    541|2014-12-25 20:27:00|2014-12-25 20:32:00|
|    283|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    519|2014-12-25 19:56:00|2014-12-25 20:01:00|
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|
|    495|2014-12-25 18:42:00|2014-12-25 18:44:00|
|    541|2014-12-25 18:28:00|2014-12-25 18:37:00|
|    585|2014-12-25 18:27:00|2014-12-25 18:37:00|
|    574|2014-12-25 18:12:00|2014-12-25 18:21:00|
|    630|2014-12-25 18:12:00|2014-12-25 18:22:00|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|
|    290|2014-12-25 18:01:00|2014-12-25 18:15:00|
|    451|2014-12-25 17:55:00|2014-12-25 18:04:00|
|    630|2014-12-25 17:55:00|2014-12-25 17:59:00|
|    574|2014-12-25 17:54:00|2014-12-25 17:59:00|


Найдём станции через которые проехал один из велосипедов, найденных ранее.

In [197]:
spark.sql("""
SELECT trips.bike_id, trips.start_date, trips.end_date, stations.name
FROM trips INNER JOIN stations
    ON trips.start_station_id == stations.id
WHERE 
    bike_id == 583 
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0) 
    AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0)
""").show()

+-------+-------------------+-------------------+--------------+
|bike_id|         start_date|           end_date|          name|
+-------+-------------------+-------------------+--------------+
|    583|2014-12-25 19:05:00|2014-12-25 19:07:00|Market at 10th|
|    583|2014-12-25 18:05:00|2014-12-25 18:22:00|Market at 10th|
|    583|2014-12-25 12:14:00|2014-12-25 12:21:00| Market at 4th|
+-------+-------------------+-------------------+--------------+



Найдём все станции, которые попали в ту же клетку h3 координатной сетки что и станции, через которые проехал велосипед 583 25.12.2014.

Отобразим gps координаты станций в координаты h3.

In [207]:
import h3
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import StringType

H3 Grid Resolutions.

In [208]:
def latlng_to_h3(lat, lng, res):
    return h3.latlng_to_cell(lat, lng, res)

h3_udf = udf(latlng_to_h3, StringType())

In [209]:
resolution = 8
stationData.withColumn('h3', h3_udf(F.col('lat'), F.col('long'), F.lit(resolution))).createOrReplaceTempView("stations_h3")

Используя вложенный sql запрос, найдём h3 координаты станций, через который проехал велосипед 583. А затем отфильтруем поездки в рождество 2014 года, которые стартовали со станций с теми же h3 координатами, что мы нашли.

In [210]:
christmas_583_contacts = spark.sql("""
SELECT trips.bike_id, trips.start_date, stations_h3.h3, stations_h3.lat, stations_h3.long, stations_h3.name
FROM trips INNER JOIN stations_h3
    ON trips.start_station_id == stations_h3.id
WHERE 
    stations_h3.h3 in (SELECT stations_h3.h3
                            FROM trips INNER JOIN stations_h3
                                ON trips.start_station_id == stations_h3.id
                            WHERE 
                                bike_id == 583 
                                AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0) 
                                AND start_date <  make_timestamp(2014, 12, 26, 0, 0, 0))
    AND start_date > make_timestamp(2014, 12, 25, 0, 0, 0)
    AND start_date < make_timestamp(2014, 12, 26, 0, 0, 0)
ORDER BY trips.start_date
""")
christmas_583_contacts.cache()
christmas_583_contacts.show()

+-------+-------------------+---------------+------------------+-------------------+------------------+
|bike_id|         start_date|             h3|               lat|               long|              name|
+-------+-------------------+---------------+------------------+-------------------+------------------+
|    439|2014-12-25 01:40:00|8828308281fffff|37.776619000000004|-122.41738500000001|    Market at 10th|
|    659|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    465|2014-12-25 09:49:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    583|2014-12-25 12:14:00|88283082abfffff|         37.786305|-122.40496599999999|     Market at 4th|
|    479|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    331|2014-12-25 12:22:00|88283082abfffff|37.781752000000004|-122.40512700000001|     5th at Howard|
|    330|2014-12-25 12:27:00|88283082abfffff|37.781752000000004|

In [212]:
import pandas as pd

In [213]:
h3_places = christmas_583_contacts.select('lat','long', 'name', 'h3').toPandas()

In [215]:
import folium

In [232]:
def init_map(hexagons, width=1100, height=900):
    lats = []
    longs = []
    for hexagon in hexagons:
        lat, long = h3.cell_to_latlng(hexagon)
        lats.append(lat)
        longs.append(long)
    return folium.Map(location=[sum(lats)/len(lats), sum(longs)/len(longs)], zoom_start=15, tiles='cartodbpositron', width=width, height=height)

def visualize_hexagons(folium_map, hexagons, color="red"):
    polylines = []

    for hex in hexagons:
        boundary = h3.cell_to_boundary(hex)

        boundary = list(boundary)  # FIX: tuple -> list

        polyline = boundary + [boundary[0]]
        polylines.append(polyline)

    for polyline in polylines:
        folium_map.add_child(
            folium.PolyLine(
                locations=polyline,
                weight=8,
                color=color
            )
        )

    return folium_map

def visualize_stations(folium_map, stations, color="red"):
    """
    stations is a dataframe with columns: lat, long, station_name
    """
    for idx, lat, long, station_name in stations.itertuples():
        folium_map.add_child(folium.map.Marker(location=(lat, long)))
        folium_map.add_child(folium.map.Marker(location=(lat, long), 
                        icon=folium.features.DivIcon(
                          icon_size=(500,36),
                          icon_anchor=(-17,37),
                          html=f'<div style="display: inline-block;font-size: 10pt; background: rgba(255, 255, 255, 0.8)">{station_name}</div>',
        )))
    return folium_map

In [233]:
m = init_map(h3_places.h3.unique())
visualize_hexagons(m, h3_places.h3.unique(), color="black")
visualize_stations(m, h3_places.loc[:, ['lat', 'long', 'name']])
display(m)

In [244]:
sc.stop()

## 2. Проведите анализ данных велопарковок на языке Python в неинтерактивном режиме (`--deploy-mode cluster`).

### 6403_KuchumovAA_BigData_Lab1.py - для анализа данных велопарковок на языке Python в неинтерактивном режиме (локально или на кластере).

```python
import argparse
import os
from datetime import datetime
from typing import NamedTuple

from pyspark import SparkConf, SparkContext


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--master", default=os.environ.get("SPARK_MASTER", "local[*]"))
    p.add_argument("--trips", type=str, default=None)
    p.add_argument("--stations", type=str, default=None)
    p.add_argument(
        "positional",
        nargs="*",
        help="Для spark-submit: master trips_path stations_path",
    )
    return p.parse_args()


args = parse_args()
pos = args.positional
master = args.master
trips_arg = args.trips
stations_arg = args.stations
if len(pos) >= 3:
    master, trips_arg, stations_arg = pos[0], pos[1], pos[2]
elif len(pos) == 2 and trips_arg is None:
    trips_arg, stations_arg = pos[0], pos[1]
if not trips_arg or not stations_arg:
    here = os.path.dirname(os.path.abspath(__file__))
    root = os.path.abspath(os.path.join(here, "..", "data"))
    trips_arg = trips_arg or os.path.join(root, "trips.csv")
    stations_arg = stations_arg or os.path.join(root, "stations.csv")

conf = SparkConf().setAppName("Lab1_Script").setMaster(master)
sc = SparkContext(conf=conf)

tripData = sc.textFile(trips_arg)
# запомним заголовок, чтобы затем его исключить из данных
tripsHeader = tripData.first()
trips = tripData.filter(lambda row: row != tripsHeader).map(lambda row: row.split(",", -1))

stationData = sc.textFile(stations_arg)
stationsHeader = stationData.first()
stations = stationData.filter(lambda row: row != stationsHeader).map(lambda row: row.split(",", -1))


def initStation(stations):
    class Station(NamedTuple):
        station_id: int
        name: str
        lat: float
        long: float
        dockcount: int
        landmark: str
        installation: str
    
    for station in stations:
        yield Station(
            station_id = int(station[0]),
            name = station[1],
            lat = float(station[2]),
            long = float(station[3]),
            dockcount = int(station[4]),
            landmark = station[5],
            installation = datetime.strptime(station[6], '%m/%d/%Y')
        )


def initTrip(trips):
    class Trip(NamedTuple):
        trip_id: int
        duration: int
        start_date: datetime
        start_station_name: str
        start_station_id: int
        end_date: datetime
        end_station_name: str
        end_station_id: int
        bike_id: int
        subscription_type: str
        zip_code: str
        
    for trip in trips:
        try:
            if not trip[1].strip() or not trip[2].strip():
                continue
            yield Trip(
                trip_id=int(trip[0]),
                duration=int(trip[1]),
                start_date=datetime.strptime(trip[2].strip(), "%m/%d/%Y %H:%M"),
                start_station_name=trip[3],
                start_station_id=int(trip[4]),
                end_date=datetime.strptime(trip[5].strip(), "%m/%d/%Y %H:%M"),
                end_station_name=trip[6],
                end_station_id=int(trip[7]),
                bike_id=int(trip[8]),
                subscription_type=trip[9],
                zip_code=trip[10],
            )
        except (ValueError, IndexError):
            continue


stationsInternal = stations.mapPartitions(initStation)
tripsInternal = trips.mapPartitions(initTrip)

tripsByStartStation = tripsInternal.keyBy(lambda trip: trip.start_station_name)


def seqFunc(acc, duration):
    duration_sum, count = acc
    return (duration_sum + duration, count + 1)


def combFunc(acc1, acc2):
    duration_sum1, count1 = acc1
    duration_sum2, count2 = acc2
    return (duration_sum1+duration_sum2, count1+count2)


def meanFunc(acc):
    duration_sum, count = acc
    return duration_sum/count


avgDurationByStartStation = tripsByStartStation\
  .mapValues(lambda trip: trip.duration)\
  .aggregateByKey(
    zeroValue=(0,0),
    seqFunc=seqFunc,
    combFunc=combFunc)\
  .mapValues(meanFunc)\
  .sortBy(lambda x: x[1], ascending=False)

#
# первые поездки стартовых станций 
#

firstStationsTrip = tripsByStartStation\
  .reduceByKey(lambda tripA, tripB: tripA if tripA.start_date < tripB.start_date else tripB)

#
# старт выполнения задач и сохранение результата в HDFS
#

avgDurationByStartStation.saveAsTextFile("avg_duration_of_start_stations")
firstStationsTrip.saveAsTextFile("first_station_trips")

# результаты сохранятся в директории распределённой файловой системы в домашней папке пользователя /user/$USER/:
#  - avg_duration_of_start_stations
#  - first_station_trips
# 
# получить результаты в виде одного файла можно командой `hadoop fs -getmerge`:
#   hadoop fs -getmerge avg_duration_of_start_stations avg_duration_of_start_stations.txt
#   hadoop fs -getmerge first_station_trips first_station_trips.txt

sc.stop()
```

## 3. Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

In [245]:
from __future__ import annotations

import argparse
import math
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Iterable, Optional, Tuple

from pyspark import SparkConf, SparkContext

if sys.platform == "win32":
    _py = os.environ.get("PYSPARK_PYTHON") or sys.executable
    os.environ.setdefault("PYSPARK_PYTHON", _py)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", _py)

In [246]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    a = min(1.0, max(0.0, a))
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))


def split_csv_header(sc, path):
    lines = sc.textFile(path)
    h = lines.first()
    rows = lines.filter(lambda r: r != h).map(lambda r: r.split(",", -1))
    return h, rows


def parse_duration_bike(row):
    try:
        if not row[1].strip():
            return None
        return int(row[8]), int(row[1])
    except:
        return None


def parse_station(row):
    try:
        return int(row[0]), float(row[2]), float(row[3]), row[1]
    except:
        return None


def parse_trip_chrono(row):
    try:
        if not row[2].strip():
            return None
        sd = datetime.strptime(row[2].strip(), "%m/%d/%Y %H:%M")
        return int(row[8]), sd, row[3], row[6], int(row[1] or 0)
    except:
        return None


def parse_zip_duration(row):
    try:
        z = row[10].strip()
        if not z or not z.isdigit():
            return None
        return z, int(row[1] or 0)
    except:
        return None


def build_station_path(ordered):
    path = []
    for _, start, end in sorted(ordered, key=lambda x: x[0]):
        if not path:
            path.append(start)
        elif path[-1] != start:
            path.append(start)
        path.append(end)
    return path

In [247]:
data_dir = Path("/mnt/c/Users/artem/Documents/BigData/Lab1/data")

trips_path = str(data_dir / "trips.csv")
stations_path = str(data_dir / "stations.csv")

conf = SparkConf().setAppName("L1_bike_tasks").setMaster("local[*]")
sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

In [248]:
_, trips = split_csv_header(sc, trips_path)
_, stations = split_csv_header(sc, stations_path)

### 1.	Найти велосипед с максимальным временем пробега.

In [249]:
bike_durations = trips.map(parse_duration_bike).filter(lambda x: x)
bike_totals = bike_durations.reduceByKey(lambda a, b: a + b)

max_bike = bike_totals.reduce(lambda a, b: a if a[1] >= b[1] else b)
max_bike

(535, 18611693)

### 2.	Найти наибольшее геодезическое расстояние между станциями.

In [250]:
st_parsed = stations.map(parse_station).filter(lambda x: x)
pairs = st_parsed.cartesian(st_parsed).filter(lambda ab: ab[0][0] < ab[1][0])

dist_rdd = pairs.map(lambda ab: (
    haversine_km(ab[0][1], ab[0][2], ab[1][1], ab[1][2]),
    ab[0], ab[1]
))

best = dist_rdd.reduce(lambda x, y: x if x[0] >= y[0] else y)
best

(69.9208759542826,
 (16, 37.333954999999996, -121.877349, 'SJSU - San Salvador at 9th'),
 (60, 37.80477, -122.40323400000001, 'Embarcadero at Sansome'))

### 3.	Найти путь велосипеда с максимальным временем пробега через станции.

In [251]:
target_bike = max_bike[0]

chrono = trips.map(parse_trip_chrono).filter(lambda x: x and x[0] == target_bike)

collected = chrono.map(lambda x: (x[1], x[2], x[3])).collect()

build_station_path(collected)

['Post at Kearney',
 'San Francisco Caltrain (Townsend at 4th)',
 'San Francisco Caltrain 2 (330 Townsend)',
 'Market at Sansome',
 '2nd at South Park',
 '2nd at Townsend',
 'Davis at Jackson',
 'San Francisco City Hall',
 'Civic Center BART (7th at Market)',
 'Post at Kearney',
 'Embarcadero at Sansome',
 'Washington at Kearney',
 'Market at Sansome',
 'Market at Sansome',
 '2nd at Folsom',
 '2nd at Townsend',
 'Temporary Transbay Terminal (Howard at Beale)',
 '2nd at Townsend',
 'Embarcadero at Sansome',
 'Clay at Battery',
 'Harry Bridges Plaza (Ferry Building)',
 'Clay at Battery',
 'San Francisco Caltrain (Townsend at 4th)',
 'Steuart at Market',
 '2nd at Townsend',
 'Harry Bridges Plaza (Ferry Building)',
 'Townsend at 7th',
 'San Francisco Caltrain (Townsend at 4th)',
 'San Francisco Caltrain 2 (330 Townsend)',
 'San Francisco Caltrain 2 (330 Townsend)',
 'Townsend at 7th',
 'Steuart at Market',
 'San Francisco Caltrain (Townsend at 4th)',
 '2nd at South Park',
 '5th at Howard',

### 4.	Найти количество велосипедов в системе.

In [252]:
trips.map(lambda r: r[8]).distinct().count()

700

### 5.	Найти пользователей потративших на поездки более 3 часов.

In [253]:
THREE_HOURS = 3 * 3600

zip_totals = (
    trips.map(parse_zip_duration)
    .filter(lambda x: x)
    .reduceByKey(lambda a, b: a + b)
)

heavy = zip_totals.filter(lambda kv: kv[1] > THREE_HOURS)

heavy.take(20)

[('95060', 758576),
 ('95112', 12742370),
 ('94041', 6276284),
 ('94117', 6901313),
 ('94402', 3303647),
 ('94102', 19128021),
 ('94612', 1860796),
 ('94609', 2503133),
 ('94158', 6248167),
 ('94133', 21637675),
 ('94597', 680583),
 ('94121', 2363342),
 ('95118', 1401452),
 ('94610', 3630628),
 ('95136', 1114542),
 ('2142', 15710),
 ('94703', 1704079),
 ('95070', 717380),
 ('94404', 3589350),
 ('94518', 481470)]

In [254]:
sc.stop()